In [1]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv("data-task-10-insurance.csv")  
print("Shape:", df.shape)
df.head()

Shape: (1338, 7)


,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1338 non-null   int64  
 1   sex       1338 non-null   str    
 2   bmi       1338 non-null   float64
 3   children  1338 non-null   int64  
 4   smoker    1338 non-null   str    
 5   region    1338 non-null   str    
 6   charges   1338 non-null   float64
dtypes: float64(2), int64(2), str(3)
memory usage: 73.3 KB


In [5]:
print("Missing values:\n", df.isnull().sum())
print("\nDuplicates:", df.duplicated().sum())

Missing values:
 age         0
sex         0
bmi         0
children    0
smoker      0
region      0
charges     0
dtype: int64

Duplicates: 1


In [6]:
df = df.drop_duplicates().reset_index(drop=True)
print("Shape after dropping duplicates:", df.shape)

Shape after dropping duplicates: (1337, 7)


In [7]:
for col in ["bmi", "charges"]:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    n_outliers = df[(df[col] < lower) | (df[col] > upper)].shape[0]
    print(f"{col}: {n_outliers} potential outliers outside [{lower:.2f}, {upper:.2f}]")

bmi: 9 potential outliers outside [13.67, 47.32]
charges: 139 potential outliers outside [-13120.72, 34524.78]


In [8]:
from sklearn.preprocessing import LabelEncoder

In [9]:
le_sex = LabelEncoder()
le_smoker = LabelEncoder()

df["sex"] = le_sex.fit_transform(df["sex"])       # female=0, male=1
df["smoker"] = le_smoker.fit_transform(df["smoker"])  # no=0, yes=1

print("sex classes:", le_sex.classes_)
print("smoker classes:", le_smoker.classes_)
df.head()

sex classes: ['female' 'male']
smoker classes: ['no' 'yes']


,age,sex,bmi,children,smoker,region,charges
0,19,0,27.900,0,1,southwest,16884.92400
1,18,1,33.770,1,0,southeast,1725.55230
2,28,1,33.000,3,0,southeast,4449.46200
3,33,1,22.705,0,0,northwest,21984.47061
4,32,1,28.880,0,0,northwest,3866.85520


In [10]:
df = pd.get_dummies(df, columns=["region"], prefix="region", drop_first=True)

region_cols = [c for c in df.columns if c.startswith("region_")]
df[region_cols] = df[region_cols].astype(int)

print(df.columns.tolist())
df.head()

['age', 'sex', 'bmi', 'children', 'smoker', 'charges', 'region_northwest', 'region_southeast', 'region_southwest']


,age,sex,bmi,children,smoker,charges,region_northwest,region_southeast,region_southwest
0,19,0,27.900,0,1,16884.92400,0,0,1
1,18,1,33.770,1,0,1725.55230,0,1,0
2,28,1,33.000,3,0,4449.46200,0,1,0
3,33,1,22.705,0,0,21984.47061,1,0,0
4,32,1,28.880,0,0,3866.85520,1,0,0


In [11]:
from sklearn.model_selection import train_test_split


In [12]:
X = df.drop(columns=["charges"])
y = df["charges"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

X_train shape: (1069, 8)
X_test shape: (268, 8)


In [ ]:
from sklearn.preprocessing import StandardScaler

In [15]:
numeric_cols = ["age", "bmi", "children"]

scaler = StandardScaler()

X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])

X_train.head()

,age,sex,bmi,children,smoker,region_northwest,region_southeast,region_southwest
1113,-1.157680,1,-0.996928,-0.907908,0,0,0,0
967,-1.300619,1,-0.792762,0.766904,0,0,0,0
598,0.914926,0,1.154664,0.766904,0,1,0,0
170,1.701087,1,1.806837,-0.907908,0,0,1,0
275,0.557580,0,-0.651417,0.766904,0,0,0,0
